# Phase 0: Compliance Foundations & System Architecture

**Time Estimate:** 6-8 hours | **Prerequisites:** None — this is where you start

---

## What This Notebook Covers

Before writing a single line of code, you need to deeply understand:
1. **What compliance actually means** in regulated environments (not textbook — real-world)
2. **NIST 800-53 Rev 5** — the control framework your tool maps everything to
3. **FedRAMP** — why it exists, how authorization works, and why small defense contractors hate it
4. **The business problem** — what auditors actually want and why current solutions fail
5. **System architecture** — designing the Compliance Evidence Collector from first principles

### Cross-References to Your Study Materials
| Topic | DDIA (Kleppmann) | System Design Interview (Xu) | DVA-C02 |
|-------|-----------------|------------------------------|----------|
| Data models for compliance findings | Ch. 2: Data Models and Query Languages | Ch. 6: Key-Value Store Design | Domain 2: Development with AWS Services |
| Reliability and fault tolerance | Ch. 1: Reliable, Scalable, Maintainable Apps | Ch. 1: Scale from Zero to Millions | Domain 3: Deployment |
| Event-driven architecture | Ch. 11: Stream Processing | Ch. 11: Design a News Feed | Domain 2: Lambda, EventBridge |
| Storage and encoding | Ch. 3: Storage and Retrieval, Ch. 4: Encoding | Ch. 7: Unique ID Generator | Domain 2: DynamoDB, S3 |

---

## Section 1: The Compliance Landscape — Why This Tool Needs to Exist

### 1.1 What Is Compliance, Really?

Compliance is **provable adherence to a set of controls**. That word "provable" is everything. It's not enough to *be* secure — you must *demonstrate* it with **evidence artifacts** that an auditor can independently verify.

Think of it like this:
- **Security** = locking your doors at night
- **Compliance** = having a timestamped video log proving every door was locked, every night, for the past 365 days

The gap between "doing security" and "proving compliance" is where hundreds of engineering hours get burned every audit cycle.

### 1.2 The Audit Pain You Lived At Owl and Carolina Cyber

You've seen this firsthand. Here's the typical audit cycle:

1. **Control identification** (weeks 1-2): Auditor sends a request list (PBC — Prepared By Client) mapping to specific controls
2. **Evidence collection** (weeks 3-8): Your team manually screenshots consoles, exports CSVs, writes narrative descriptions
3. **Evidence review** (weeks 9-12): Auditor reviews, finds gaps, requests more evidence, back-and-forth
4. **Remediation** (weeks 13-16): Fix findings, re-collect evidence for remediated items
5. **Final report** (weeks 17-20): Auditor issues opinion/authorization

**The bottleneck is step 2.** A single SOC 2 audit might require 200+ individual evidence artifacts. FedRAMP can require 400+. Each one must be:
- Timestamped
- Attributed to a specific resource
- Mapped to a specific control
- Freshly generated (not stale)
- In a format the auditor accepts

### 1.3 The Market Opportunity

| Company | Raised | Annual Revenue | What They Do |
|---------|--------|----------------|---------------|
| Drata | $328M | ~$100M ARR | Continuous compliance automation |
| Vanta | $203M | ~$100M ARR | Automated security compliance |
| Secureframe | $79M | ~$30M ARR | Compliance automation platform |
| Tugboat Logic (Acquired) | $18M | Acquired by OneTrust | Risk & compliance automation |

**But here's the gap:** These are all broad-market GRC platforms charging $15K-$50K/year. Small defense contractors (50-200 employees) with FedRAMP requirements often can't justify that cost but spend $100K+ in consultant hours per authorization cycle.

**Your angle:** A focused, AWS-native tool specifically for FedRAMP compliance evidence collection, priced at $5K-$10K/year, sold by someone with TS/SCI clearance who speaks their language.

### Documentation & Further Reading
- [Drata's IPO S-1 when filed — study their metrics](https://www.drata.com)
- [Vanta's compliance automation approach](https://www.vanta.com/resources)
- [FedRAMP Marketplace — see who's authorized and for what](https://marketplace.fedramp.gov/products)

## Section 2: NIST 800-53 Rev 5 — The Control Framework Deep Dive

### 2.1 What Is NIST 800-53?

NIST Special Publication 800-53 Revision 5 is the **catalog of security and privacy controls** for federal information systems. Published by the National Institute of Standards and Technology, it's the backbone of FedRAMP, FISMA, and most federal compliance regimes.

**Key concept:** NIST 800-53 is a *catalog* — it lists controls you *could* implement. A *baseline* (like FedRAMP Low, Moderate, or High) tells you which controls you *must* implement.

### 2.2 The 20 Control Families

Each control family is identified by a two-letter code. Here's every family with what matters for AWS automation:

| ID | Family | # Controls | AWS Automatable? | Key AWS Services |
|----|--------|-----------|------------------|------------------|
| AC | Access Control | 25 | HIGH | IAM, Organizations, SSO |
| AT | Awareness & Training | 6 | LOW | Manual process |
| AU | Audit & Accountability | 16 | HIGH | CloudTrail, CloudWatch, Config |
| CA | Assessment, Authorization & Monitoring | 9 | MEDIUM | Security Hub, Inspector |
| CM | Configuration Management | 14 | HIGH | Config, Systems Manager |
| CP | Contingency Planning | 13 | MEDIUM | Backup, DR configurations |
| IA | Identification & Authentication | 12 | HIGH | IAM, Cognito, SSO |
| IR | Incident Response | 10 | MEDIUM | GuardDuty, Detective, EventBridge |
| MA | Maintenance | 6 | LOW | Systems Manager Patch Manager |
| MP | Media Protection | 8 | MEDIUM | EBS encryption, S3 policies |
| PE | Physical & Environmental | 23 | NONE | AWS handles (shared responsibility) |
| PL | Planning | 11 | LOW | Documentation process |
| PM | Program Management | 32 | LOW | Organizational process |
| PS | Personnel Security | 9 | NONE | HR process |
| PT | PII Processing & Transparency | 8 | LOW | Macie, data classification |
| RA | Risk Assessment | 10 | MEDIUM | Inspector, GuardDuty |
| SA | System & Services Acquisition | 23 | LOW | Procurement process |
| SC | System & Communications Protection | 51 | HIGH | VPC, KMS, ACM, WAF |
| SI | System & Information Integrity | 23 | HIGH | GuardDuty, Inspector, Patch Manager |
| SR | Supply Chain Risk Management | 12 | LOW | Process + some automation |

**Key insight for your tool:** Focus on the HIGH automatable families first: **AC, AU, CM, IA, SC, SI**. These 6 families cover ~145 controls and are where 80% of the audit evidence collection pain lives.

### 2.3 Anatomy of a Control

Let's dissect a real control to understand the data model:

```
Control ID: AC-2 — Account Management
├── Control Text: "Define and document the types of accounts allowed and
│   specifically prohibited for use within the system..."
├── Control Enhancements:
│   ├── AC-2(1): Automated Account Management
│   ├── AC-2(2): Automated Temporary/Emergency Account Management  
│   ├── AC-2(3): Disable Accounts
│   ├── AC-2(4): Automated Audit Actions
│   └── ... (13 total enhancements)
├── FedRAMP Baseline:
│   ├── Low: AC-2 base only
│   ├── Moderate: AC-2, AC-2(1), AC-2(2), AC-2(3), AC-2(4), AC-2(5)
│   └── High: All enhancements
├── Evidence an Auditor Wants:
│   ├── List of all user accounts with creation dates
│   ├── IAM policies showing least-privilege assignment
│   ├── Proof of account reviews (quarterly)
│   ├── Disabled/removed accounts list with dates
│   └── Automated alerts for suspicious account activity
└── AWS Services That Provide This Evidence:
    ├── IAM: ListUsers, GetCredentialReport, ListPolicies
    ├── CloudTrail: Account activity logs
    ├── Config: iam-user-unused-credentials-check rule
    └── Security Hub: IAM findings from CIS benchmark
```

**This structure is what your tool must automate.** For each control, you need to:
1. Know which AWS API calls provide evidence
2. Format the response into audit-ready output
3. Map findings to the specific control ID
4. Timestamp everything

### 2.4 FedRAMP Baselines — What You'll Implement

FedRAMP defines three impact levels, each selecting a different set of NIST 800-53 controls:

| Baseline | # Controls | Use Case | Your Target Market |
|----------|-----------|----------|--------------------|
| FedRAMP Low | ~156 | Public data, minimal impact | Small SaaS tools |
| FedRAMP Moderate | ~325 | Controlled unclassified info (CUI) | **PRIMARY TARGET** |
| FedRAMP High | ~421 | Law enforcement, emergency services | Larger contractors |

**Start with FedRAMP Moderate.** That's where 90% of defense contractors live. They're handling CUI (Controlled Unclassified Information) and need Moderate authorization.

### Documentation & Further Reading
- [NIST 800-53 Rev 5 Full Catalog (official)](https://csrc.nist.gov/publications/detail/sp/800-53/rev-5/final)
- [NIST 800-53 Control Database (searchable)](https://csf.tools/reference/nist-sp-800-53/r5/)
- [FedRAMP Control Baselines (official spreadsheets)](https://www.fedramp.gov/documents-templates/)
- [FedRAMP Moderate Baseline Control List](https://www.fedramp.gov/assets/resources/documents/FedRAMP_Security_Controls_Baseline.xlsx)
- [NIST Cybersecurity Framework crosswalk to 800-53](https://csrc.nist.gov/Projects/cybersecurity-framework/nist-cybersecurity-framework-and-nist-sp-800-53-rev-5)

## Section 3: AWS Shared Responsibility & Security Services Primer

### 3.1 Shared Responsibility Model

**DVA-C02 Exam Alert:** This is one of the most tested concepts. Know it cold.

```
┌─────────────────────────────────────────────────────────────┐
│                    CUSTOMER RESPONSIBILITY                   │
│              "Security IN the Cloud"                         │
│  ┌─────────────────────────────────────────────────────────┐│
│  │ Customer Data                                           ││
│  │ Platform, Applications, IAM                             ││
│  │ Operating System, Network, Firewall Config              ││
│  │ Client-side Encryption, Server-side Encryption          ││
│  │ Network Traffic Protection                              ││
│  └─────────────────────────────────────────────────────────┘│
├─────────────────────────────────────────────────────────────┤
│                    AWS RESPONSIBILITY                        │
│              "Security OF the Cloud"                         │
│  ┌─────────────────────────────────────────────────────────┐│
│  │ Hardware / AWS Global Infrastructure                    ││
│  │ Regions, Availability Zones, Edge Locations             ││
│  │ Compute, Storage, Database, Networking (physical)       ││
│  │ Software: Hypervisor, OS of managed services            ││
│  └─────────────────────────────────────────────────────────┘│
└─────────────────────────────────────────────────────────────┘
```

**Why this matters for compliance:** For FedRAMP, AWS has its own FedRAMP High authorization. That covers the PE (Physical) and parts of other control families. But the *customer* is responsible for everything in their account — IAM policies, security groups, encryption settings, logging configs. **Your tool automates evidence collection for the customer-responsibility side.**

### 3.2 The Six AWS Security Services Your Tool Relies On

#### AWS Security Hub
**What it does:** Aggregates security findings from multiple AWS services and third-party tools into a single pane. Runs automated compliance checks against standards (CIS, NIST, PCI DSS).

**Why you need it:** It's the primary data source for your tool. Security Hub already maps findings to NIST 800-53 controls via the AWS FSBP (Foundational Security Best Practices) standard.

**Key API calls you'll use:**
```python
# These are the boto3 calls you'll implement in Phase 1
securityhub.get_findings()              # Pull all findings
securityhub.get_enabled_standards()     # Which standards are active
securityhub.describe_standards_controls() # Get control-level detail
securityhub.batch_update_findings()     # Update finding status
```

**DVA-C02 connection:** Security Hub integrates with EventBridge for event-driven architectures (Domain 2 topic).

**Docs:** [Security Hub User Guide](https://docs.aws.amazon.com/securityhub/latest/userguide/what-is-securityhub.html) | [Security Hub API Reference](https://docs.aws.amazon.com/securityhub/1.0/APIReference/Welcome.html)

---

#### AWS Config
**What it does:** Continuously records AWS resource configurations and evaluates them against rules. Think of it as a DVR for your infrastructure state.

**Why you need it:** Config provides the raw evidence — "At timestamp X, resource Y had configuration Z." That's exactly what auditors want.

**DDIA Connection (Ch. 3 — Storage and Retrieval):** Config is essentially an append-only log of configuration changes — the same concept as a write-ahead log or event sourcing pattern Kleppmann describes. Each configuration item is a point-in-time snapshot, and you can query the history to get any past state.

**Key API calls:**
```python
config.get_compliance_details_by_config_rule()  # Rule compliance
config.get_resource_config_history()             # Historical configs
config.select_resource_config()                  # SQL-like queries
config.describe_compliance_by_config_rule()      # Summary compliance
```

**Docs:** [AWS Config Developer Guide](https://docs.aws.amazon.com/config/latest/developerguide/WhatIsConfig.html) | [AWS Config Managed Rules](https://docs.aws.amazon.com/config/latest/developerguide/managed-rules-by-aws-config.html)

---

#### AWS CloudTrail
**What it does:** Logs every API call made in your AWS account. Who did what, when, from where.

**Why you need it:** Controls in the AU (Audit) family require proof of logging. CloudTrail IS the audit log.

**Key API calls:**
```python
cloudtrail.lookup_events()           # Search recent events
cloudtrail.describe_trails()         # Trail configuration
cloudtrail.get_trail_status()        # Is logging active?
# For historical: query CloudTrail Lake or Athena against S3 logs
```

**Docs:** [CloudTrail User Guide](https://docs.aws.amazon.com/awscloudtrail/latest/userguide/cloudtrail-user-guide.html)

---

#### AWS IAM
**What it does:** Identity and Access Management — controls who can do what in your AWS account.

**Why you need it:** AC (Access Control) and IA (Identification & Authentication) families are heavily IAM-dependent. You need to pull credential reports, policy details, and access patterns.

**Key API calls:**
```python
iam.generate_credential_report()     # Account-wide credential status
iam.get_credential_report()          # Download the report
iam.list_users()                     # All IAM users
iam.list_attached_user_policies()    # Policies per user
iam.get_account_authorization_details()  # Everything in one call
iam.get_account_password_policy()    # Password policy settings
```

**DVA-C02 Exam Alert:** IAM is tested heavily. Know policies (identity-based vs resource-based), roles, trust policies, and the principle of least privilege.

**Docs:** [IAM User Guide](https://docs.aws.amazon.com/IAM/latest/UserGuide/introduction.html) | [IAM API Reference](https://docs.aws.amazon.com/IAM/latest/APIReference/Welcome.html)

---

#### AWS GuardDuty
**What it does:** Threat detection using ML and threat intelligence. Monitors CloudTrail, VPC Flow Logs, and DNS logs for suspicious activity.

**Why you need it:** IR (Incident Response) and SI (System Integrity) controls require threat detection capabilities.

**Key API calls:**
```python
guardduty.list_findings()            # Get finding IDs
guardduty.get_findings()             # Full finding details
guardduty.get_detector()             # Detector configuration
```

**Docs:** [GuardDuty User Guide](https://docs.aws.amazon.com/guardduty/latest/ug/what-is-guardduty.html)

---

#### Amazon Inspector
**What it does:** Automated vulnerability assessment for EC2 instances, Lambda functions, and container images.

**Why you need it:** RA (Risk Assessment) and SI controls require vulnerability scanning evidence.

**Key API calls:**
```python
inspector2.list_findings()           # Vulnerability findings
inspector2.list_coverage()           # What's being scanned
inspector2.get_findings_report_status()  # Report generation
```

**Docs:** [Amazon Inspector User Guide](https://docs.aws.amazon.com/inspector/latest/user/what-is-inspector.html)

## Section 4: System Architecture — Designing the Compliance Evidence Collector

### 4.1 Design Principles

Before drawing boxes and arrows, let's establish the principles that drive our architecture decisions:

1. **Serverless-first:** Small defense contractors don't have dedicated ops teams. No servers to manage = lower total cost = easier sale.
2. **Event-driven:** Compliance findings change in real-time. Our tool should react, not poll on a schedule only.
3. **Idempotent:** Running the collector twice should produce the same result. Critical for audit credibility.
4. **Auditable itself:** The tool that collects compliance evidence must itself be auditable. Dog-fooding.
5. **Multi-tenant ready:** Even if you start single-tenant, design for multi-tenant from day one. Your SaaS future depends on it.

**DDIA Connection (Ch. 1):** Kleppmann's three concerns — Reliability, Scalability, Maintainability — map directly:
- *Reliability:* Evidence collection must not silently fail; a missing evidence artifact is an audit finding
- *Scalability:* A single AWS account might have 10,000 resources; need to handle large environments
- *Maintainability:* NIST controls get updated; AWS adds new services; your mapping layer must evolve

**System Design Interview Connection (Ch. 1-2):** This is a classic system design problem — ingesting data from many sources, processing it, storing it, and presenting it. Think about the back-of-envelope estimation: How many findings per account? How large is each finding? What's the read/write ratio?

### 4.2 Back-of-Envelope Estimation

**System Design Interview technique** — always do this before designing:

```
Assumptions:
- Average AWS account: ~500 resources
- Security Hub findings per account: ~200-2,000
- Each finding: ~2KB JSON
- Config rules per account: ~150 managed rules
- Config compliance items: ~500-5,000
- Each compliance item: ~1KB JSON
- Scan frequency: Every 6 hours (4x daily)
- Evidence PDF: ~50KB per control family, ~20 families = 1MB total

Daily data per account:
- Findings: 2,000 × 2KB × 4 scans = 16MB
- Config: 5,000 × 1KB × 4 scans = 20MB
- PDFs: 1MB × 4 = 4MB
- Total: ~40MB/day/account

For 100 customers:
- Daily: 4GB
- Monthly: 120GB
- Yearly: 1.4TB

S3 storage cost: ~$0.023/GB = ~$32/month for 100 customers
Lambda invocations: 100 accounts × 4 scans × ~50 Lambda calls = 20,000/day
Lambda cost: Negligible at this scale (~$0.20/day)
```

**Conclusion:** This is a very affordable system to run. Your margins on a $5K-$10K/year product would be enormous. The value isn't in compute — it's in the **compliance knowledge encoded in the control mappings**.

### 4.3 Architecture Diagram — Phase by Phase

```
PHASE 1: Data Collection
========================
                                    ┌──────────────────┐
┌─────────────┐  EventBridge  ┌────▶│ findings_table   │
│ Security Hub │──(schedule)──▶│    │ (DynamoDB)       │
└─────────────┘               │    └──────────────────┘
┌─────────────┐               │    ┌──────────────────┐
│ AWS Config   │──────────────┤───▶│ raw_evidence/    │
└─────────────┘               │    │ (S3 bucket)      │
┌─────────────┐               │    └──────────────────┘
│ IAM          │──────────────┤
└─────────────┘      ┌────────┴────────┐
┌─────────────┐      │  collector_     │
│ CloudTrail   │─────▶│  lambda        │
└─────────────┘      │  (Python 3.12)  │
┌─────────────┐      └─────────────────┘
│ GuardDuty    │──────────────┘
└─────────────┘


PHASE 2: Control Mapping
========================
┌──────────────────┐    ┌───────────────────┐    ┌──────────────────┐
│ findings_table   │───▶│ mapping_lambda    │───▶│ control_status   │
│ (raw findings)   │    │                   │    │ (DynamoDB)       │
└──────────────────┘    │ NIST 800-53       │    └──────────────────┘
                        │ mapping rules     │
┌──────────────────┐    │ (JSON config)     │    ┌──────────────────┐
│ raw_evidence/    │───▶│                   │───▶│ mapped_evidence/ │
│ (S3)             │    └───────────────────┘    │ (S3)             │
└──────────────────┘                             └──────────────────┘


PHASE 3: Evidence Generation
============================
┌──────────────────┐    ┌───────────────────┐    ┌──────────────────┐
│ control_status   │───▶│ pdf_generator_    │───▶│ evidence_pdfs/   │
│ mapped_evidence/ │    │ lambda            │    │ (S3)             │
└──────────────────┘    │                   │    └──────────────────┘
                        │ ReportLab +       │           │
                        │ Jinja2 templates  │           │
                        └───────────────────┘           │
                                                        ▼
                                               ┌──────────────────┐
                                               │ SNS notification │
                                               │ "Report ready"   │
                                               └──────────────────┘


PHASE 4: Drift Detection
========================
┌──────────────────┐    ┌───────────────────┐    ┌──────────────────┐
│ control_status   │───▶│ drift_detector_   │───▶│ drift_alerts     │
│ (current scan)   │    │ lambda            │    │ (SNS → Email)    │
├──────────────────┤    │                   │    ├──────────────────┤
│ control_status   │───▶│ Compare current   │───▶│ drift_history    │
│ (previous scan)  │    │ vs. baseline      │    │ (DynamoDB)       │
└──────────────────┘    └───────────────────┘    └──────────────────┘


PHASE 5: Dashboard
==================
┌──────────────────┐    ┌───────────────────┐    ┌──────────────────┐
│ API Gateway      │───▶│ dashboard_api_    │◀──▶│ DynamoDB tables  │
│ (REST API)       │    │ lambda            │    │ S3 buckets       │
└──────────────────┘    └───────────────────┘    └──────────────────┘
         ▲
         │
┌────────┴─────────┐
│ React SPA        │
│ (S3 + CloudFront)│
│ - Posture over   │
│   time charts    │
│ - Control status │
│ - PDF downloads  │
│ - Drift alerts   │
└──────────────────┘


PHASE 6: Infrastructure as Code
================================
All of the above deployed via Terraform modules:
├── modules/
│   ├── collector/     (Lambda + EventBridge + IAM roles)
│   ├── mapping/       (Lambda + DynamoDB tables)
│   ├── evidence/      (Lambda + S3 buckets + SNS)
│   ├── drift/         (Lambda + DynamoDB + SNS)
│   ├── dashboard/     (API Gateway + Lambda + S3 + CloudFront)
│   └── monitoring/    (CloudWatch dashboards + alarms)
└── environments/
    ├── dev/
    ├── staging/
    └── prod/
```

### 4.4 Data Flow — The Full Picture

**DDIA Connection (Ch. 11 — Stream Processing):** Our architecture is a streaming data pipeline:
1. **Producers:** AWS services generating findings (Security Hub, Config, etc.)
2. **Stream processor:** Lambda functions that transform, map, and enrich findings
3. **Consumers:** PDF generator, dashboard, drift detector
4. **Sinks:** S3 (evidence), DynamoDB (state), SNS (notifications)

This is a simplified version of the stream processing architectures in DDIA Chapter 11. Kleppmann describes how derived data (our PDFs and dashboard) should be deterministically reproducible from the source events (raw findings). That's exactly our design — you can re-run the evidence generator at any time and get the same output for the same input.

### 4.5 API Design

**System Design Interview Connection:** Always define your APIs early.

```
REST API (API Gateway → Lambda)

GET  /api/v1/scans                        # List all scan results
GET  /api/v1/scans/{scan_id}              # Get specific scan
POST /api/v1/scans                        # Trigger new scan

GET  /api/v1/controls                     # All controls with status
GET  /api/v1/controls/{control_id}        # Specific control detail
GET  /api/v1/controls/{control_id}/evidence  # Evidence for a control

GET  /api/v1/reports                      # List generated reports
GET  /api/v1/reports/{report_id}/download # Download PDF
POST /api/v1/reports                      # Generate new report

GET  /api/v1/drift                        # Drift events
GET  /api/v1/drift/summary                # Drift summary

GET  /api/v1/posture                      # Overall compliance posture
GET  /api/v1/posture/history              # Posture over time
```

## Section 5: Data Modeling — Designing the Schema

### 5.1 Choosing the Right Data Model

**DDIA Connection (Ch. 2 — Data Models and Query Languages):** Kleppmann discusses relational vs. document vs. graph models. For our compliance data:

- **Findings** are semi-structured JSON documents with varying schemas across services → **Document model** (DynamoDB or S3 JSON)
- **Control mappings** have hierarchical relationships (family → control → enhancement) → Could be relational, but we'll use **denormalized documents** for query simplicity
- **Posture history** is time-series data → **Time-series pattern** in DynamoDB or S3 partitioned by date

**DVA-C02 Exam Alert:** DynamoDB data modeling is a major exam topic. Know:
- Partition key vs. sort key design
- Single-table design patterns
- GSI (Global Secondary Index) vs. LSI (Local Secondary Index)
- Read/write capacity units and auto-scaling

### 5.2 DynamoDB Table Design

We'll use a **single-table design** (a DynamoDB best practice you should know for the exam):

```
Table: ComplianceData
──────────────────────
Partition Key (PK): String
Sort Key (SK): String

Access Patterns → Key Design:

1. Get all findings for a scan:
   PK = SCAN#{scan_id}
   SK = FINDING#{finding_id}

2. Get all controls for a scan:
   PK = SCAN#{scan_id}
   SK = CONTROL#{control_family}#{control_id}

3. Get control history over time:
   PK = CONTROL#{control_id}
   SK = SCAN#{timestamp}

4. Get drift events:
   PK = DRIFT#{date}
   SK = EVENT#{timestamp}#{control_id}

5. Get report metadata:
   PK = REPORT#{report_id}
   SK = META

GSI-1 (for querying by control family):
   PK = FAMILY#{family_code}  (e.g., FAMILY#AC)
   SK = SCAN#{timestamp}

GSI-2 (for querying by severity):
   PK = SEVERITY#{level}  (e.g., SEVERITY#HIGH)
   SK = SCAN#{timestamp}
```

**DDIA Connection (Ch. 3 — Storage and Retrieval):** This single-table design is essentially building our own index structures on top of a key-value store. The PK/SK pattern is analogous to a composite index in relational databases. The GSIs are secondary indexes. Kleppmann's discussion of LSM-trees and SSTables in Chapter 3 is the underlying storage mechanism DynamoDB uses.

### 5.3 S3 Bucket Structure

```
s3://compliance-evidence-{account_id}/
├── raw/
│   ├── security-hub/{date}/{scan_id}/findings.json
│   ├── config/{date}/{scan_id}/compliance.json
│   ├── iam/{date}/{scan_id}/credential-report.csv
│   ├── cloudtrail/{date}/{scan_id}/events.json
│   └── guardduty/{date}/{scan_id}/findings.json
├── mapped/
│   └── {date}/{scan_id}/
│       ├── AC/  (Access Control evidence)
│       ├── AU/  (Audit evidence)
│       ├── CM/  (Configuration Management evidence)
│       └── ...  (each control family)
├── reports/
│   └── {date}/{scan_id}/
│       ├── full-report.pdf
│       ├── executive-summary.pdf
│       └── by-family/
│           ├── AC-report.pdf
│           └── ...
└── drift/
    └── {date}/drift-report.json
```

**DVA-C02 Exam Alert:** Know S3 storage classes (Standard, IA, Glacier), lifecycle policies, bucket policies vs. ACLs, and server-side encryption options (SSE-S3, SSE-KMS, SSE-C).

### Documentation & Further Reading
- [DynamoDB Best Practices — Single Table Design](https://docs.aws.amazon.com/amazondynamodb/latest/developerguide/bp-general-nosql-design.html)
- [Alex DeBrie's DynamoDB Book (excellent supplement)](https://www.dynamodbbook.com/)
- [S3 Developer Guide](https://docs.aws.amazon.com/AmazonS3/latest/userguide/Welcome.html)
- [DDIA Chapter 2 summary notes](https://github.com/keyvanakbary/learning-notes/blob/master/books/designing-data-intensive-applications.md)

## Section 6: The DVA-C02 Exam Alignment

This project covers a huge portion of the DVA-C02 exam domains. Here's the explicit mapping:

### Domain 1: Development with AWS Services (32%)
| Exam Topic | Where You'll Learn It |
|------------|----------------------|
| Develop code for Lambda | Phase 1-4 (every Lambda function you build) |
| Develop code using AWS SDKs (boto3) | Phase 1 (collector), Phase 2 (mapper) |
| Interact with AWS services using APIs | Every single phase |
| DynamoDB CRUD operations | Phase 2, 4 (control status, drift history) |
| S3 operations | Phase 1, 3 (evidence storage, PDF upload) |
| API Gateway | Phase 5 (dashboard API) |

### Domain 2: Security (26%)
| Exam Topic | Where You'll Learn It |
|------------|----------------------|
| IAM policies and roles | Phase 1 (collector IAM role), Phase 6 (Terraform IAM) |
| Encrypt data at rest and in transit | Phase 3 (S3 encryption), Phase 5 (HTTPS) |
| Manage sensitive data | Phase 1 (credential reports), Phase 4 (KMS) |
| Security Hub, Config, GuardDuty | Phase 0-1 (core data sources) |

### Domain 3: Deployment (24%)
| Exam Topic | Where You'll Learn It |
|------------|----------------------|
| Deploy serverless applications | Phase 6 (Terraform + SAM) |
| CI/CD pipelines | Phase 6 (CodePipeline/GitHub Actions) |
| CloudFormation / IaC concepts | Phase 6 (Terraform modules) |
| Environment configuration | Phase 6 (dev/staging/prod) |

### Domain 4: Troubleshooting and Optimization (18%)
| Exam Topic | Where You'll Learn It |
|------------|----------------------|
| CloudWatch Logs and metrics | Phase 4, 6 (monitoring) |
| X-Ray tracing | Phase 6 (distributed tracing) |
| Lambda optimization | Phase 1-4 (cold starts, memory, timeout) |
| DynamoDB optimization | Phase 2 (query patterns, GSI design) |

### Study Resources
- [AWS DVA-C02 Exam Guide (official)](https://d1.awsstatic.com/training-and-certification/docs-dev-associate/AWS-Certified-Developer-Associate_Exam-Guide.pdf)
- [AWS Skill Builder — Free practice exam](https://explore.skillbuilder.aws/learn/course/external/view/elearning/14724/exam-prep-standard-course-aws-certified-developer-associate-dva-c02)
- [Tutorials Dojo DVA-C02 practice tests](https://tutorialsdojo.com/aws-certified-developer-associate/)
- [AWS Well-Architected Framework — Security Pillar](https://docs.aws.amazon.com/wellarchitected/latest/security-pillar/welcome.html)

## Section 7: Project Setup — What You Need Before Phase 1

### 7.1 AWS Account Setup

**Real-world note:** The commands below require admin-level IAM permissions. If you're using a scoped deploy user (e.g., a CI/CD user), you'll get `AccessDeniedException`. Configure a separate admin profile first:

```bash
aws configure --profile iamadmin
# Enter your admin user's access key, secret key, region (us-east-1), output format (json)

# Verify identity before running anything
aws sts get-caller-identity --profile iamadmin
```

```bash
# 1. Create a dedicated AWS account (or use an existing dev account)
#    DO NOT use a production account for learning

# 2. Enable Security Hub
#    PowerShell has no backslash line continuation — put it on one line
aws securityhub enable-security-hub --enable-default-standards --control-finding-generator SECURITY_CONTROL --profile iamadmin

# 3. Enable AWS Config
#    The role ARN in the docs examples is incomplete. You must first create the
#    Config service-linked role, then reference it with your actual account ID.
aws iam create-service-linked-role --aws-service-name config.amazonaws.com --profile iamadmin

aws configservice put-configuration-recorder \
  --configuration-recorder name=default,roleARN=arn:aws:iam::{YOUR_ACCOUNT_ID}:role/aws-service-role/config.amazonaws.com/AWSServiceRoleForConfig \
  --recording-group allSupported=true,includeGlobalResourceTypes=true \
  --profile iamadmin

# 4. Enable CloudTrail
#    CloudTrail requires an S3 bucket to exist with a specific bucket policy BEFORE
#    create-trail will succeed — you cannot use an arbitrary bucket name.
aws s3api create-bucket --bucket cloudtrail-logs-{YOUR_ACCOUNT_ID} --region us-east-1 --profile iamadmin

# Save cloudtrail-bucket-policy.json (file included in this repo), then apply it:
aws s3api put-bucket-policy \
  --bucket cloudtrail-logs-{YOUR_ACCOUNT_ID} \
  --policy file://cloudtrail-bucket-policy.json \
  --profile iamadmin

aws cloudtrail create-trail \
  --name compliance-trail \
  --s3-bucket-name cloudtrail-logs-{YOUR_ACCOUNT_ID} \
  --is-multi-region-trail \
  --profile iamadmin

# Trail is created paused — start logging explicitly
aws cloudtrail start-logging --name compliance-trail --profile iamadmin

# 5. Enable GuardDuty
aws guardduty create-detector --enable --profile iamadmin

# 6. Create a dev IAM user with read-only security access
#    Use this profile for day-to-day collector work; keep iamadmin for setup only.
aws iam create-user --user-name compliance-dev --profile iamadmin
aws iam attach-user-policy --user-name compliance-dev --policy-arn arn:aws:iam::aws:policy/SecurityAudit --profile iamadmin
aws iam create-access-key --user-name compliance-dev --profile iamadmin

# Run this interactively in your terminal (not via ! prefix — needs stdin):
# aws configure --profile compliance-dev

# Verify the dev profile works
aws sts get-caller-identity --profile compliance-dev
```

### 7.2 Local Development Setup

```bash
# Python environment
python -m venv compliance-collector-env
source compliance-collector-env/bin/activate  # Linux/Mac

# Core dependencies
pip install boto3 pytest moto reportlab jinja2 pandas

# Development tools
pip install black flake8 mypy ipykernel jupyter

# Terraform (for Phase 6)
# Download from https://www.terraform.io/downloads

# AWS CLI configured
aws configure
# Enter your Access Key, Secret Key, default region (us-east-1 recommended)
```

### 7.3 Project Structure

```
compliance-evidence-collector/
├── notebooks/                    # Your learning playbooks (this curriculum)
│   ├── phase-0-foundations/
│   ├── phase-1-data-collection/
│   ├── phase-2-control-mapping/
│   ├── phase-3-evidence-pdf/
│   ├── phase-4-storage-drift/
│   ├── phase-5-dashboard/
│   ├── phase-6-iac-deployment/
│   └── phase-7-demo-gtm/
├── src/                          # Production source code
│   ├── collector/                # Phase 1: Data collection lambdas
│   ├── mapper/                   # Phase 2: Control mapping engine
│   ├── evidence/                 # Phase 3: PDF generation
│   ├── drift/                    # Phase 4: Drift detection
│   ├── api/                      # Phase 5: Dashboard API
│   └── common/                   # Shared utilities
├── terraform/                    # Phase 6: Infrastructure as Code
│   ├── modules/
│   └── environments/
├── tests/                        # Unit and integration tests
├── templates/                    # PDF report templates
├── mappings/                     # NIST control mapping configs
└── docs/                         # Architecture decisions, runbooks
```

### 7.4 Estimated Cost to Run

Running this in a dev AWS account:
- Security Hub: ~$0.00 (free tier covers small environments)
- AWS Config: ~$2-5/month (recording rules)
- Lambda: ~$0.00 (free tier: 1M requests/month)
- DynamoDB: ~$0.00 (free tier: 25GB, 25 RCU/WCU)
- S3: ~$0.05/month
- **Total dev cost: ~$2-5/month**

## Section 8: Exercises & Checkpoints

Before moving to Phase 1, make sure you can answer these:

### Conceptual Questions
1. What's the difference between NIST 800-53 (the catalog) and a FedRAMP baseline (the selection)?
    NIST 800-53 is a full catalog of 1,000 plus controls across 20 families that "could" apply to my federal system. A fedramp baseline(can be low, moderate, high) is curated selection from that catalog based on the impact of the system. Fedramp moderate, for example picks ~325 of these controls and says "these are the ones you must implement". Catalog is our menu (Nist) and baseline(fedramp is my order)
2. Name the 6 NIST 800-53 control families that are most automatable in AWS. Why these?
    AC, AU, CM, IA, SC, SI - because these map directly to AWS API-querable state:
    = AC (Access Control) - IAM policies, roles, users
    = AU (Audit & Accountability ) - cloudtrail, cloudwatch logs
    = CM (Configuration Management) - AWS Config Rules
    = IA (Identification & Authentication) - IAM credntial reports, MFA status
    = SC (System & Communications Protection) - VPC configs, KMS, ACM, Security groups
    = SI (System & Information Integrity) - GuardDuty findings, Inspector vulnerabilites
3. In the Shared Responsibility Model, which NIST control families does AWS handle for you?
    PE (Physical & Environmental) entirely - AWS owns the data ceneters Partial on media disposal. everything else is customer owned This is the shared responsibility model: AWS handles security of the cloud (physical, hypervisor, hardware); you handle security in the cloud (IAM, encryption, network configs, logging).
4. Why does our tool use DynamoDB instead of RDS? (Hint: think about the data model and DDIA Ch. 2)
    would use dynamodb for its flexiblity, example security hub finding looks nothing like a config compliance item; if you put that into a relational db would have wide nullable schemas or lots of joins. DynamoDB document model lets each item carry its own shape. So when the data has a document like, self contained structure with little to no need for many to many findings simplicity of nosql wins and no schema migration risk when aws changes a findings json structure
5. What makes an evidence artifact "audit-ready"? List the four required properties.
    1. Timestamped - when was the state observed
    2. resource attributed - which specific resource does this apply to (arn, ID, name)
    3. control mapped -explicitly tied to a specific NIST control ID(AU-2)
    4. Fresh - generated within the audit window not stale screenshots from months ago

### DVA-C02 Practice Questions
1. A Lambda function needs to read from Security Hub and write to DynamoDB. How should you configure permissions? (IAM execution role with least-privilege policy)
    Lamba reading security hub and writing to DynamoDb. should create an IAM execution role and attach it to the lambda function. Needs two least privledge statements: securityhub: GetFindings (and related read actions) on the security hub resource and dynamoDB:PutItem/ dynamodb:updateitem on the specific dyanmodb table arn  Never use AdministratorAccess or broad wildcards — auditors will flag it, and it's exactly the kind of thing your own tool would catch. 
2. Your S3 bucket stores compliance evidence that must be encrypted at rest with customer-managed keys. Which encryption option should you use? (SSE-KMS with CMK)
    Server Side Encryption with customer managed keys. SSE-S3 uses AWS- managed keys (you do not control rotation or access policy). SEE KMS with CMK lets you say who can use the key via KMS Key policy, audit every decrypt via cloudtrial and rotate on your schedule. for Fedramp evidence that auditors will inspect you need that control chain visible and provable
3. You need to trigger a Lambda function every 6 hours. Which service should you use? (EventBridge scheduled rule)
    eventbridge scheduled rule using a cron or rate exprespress (rate(6hours)). Event bridge is a modern replace to cloudwatch events. the rule invokes lambda directly no queue needed for simple periodic trigger
4. Your DynamoDB table has a hot partition because all scans from today hit the same partition key. How do you fix this? (Add scan_id or timestamp to PK to distribute writes)
    add high cardinality attribute  to the partition key - SCAN#{scan_id} instead of SCAN#{date}. every scan gets a unique UUID this way writes will fan out across partitions. If you need to query all scans for a date, add a Global Secondary index with date as the GSI. the core DynamoDb trade off is design you r PK for wrtie distros use GSI for alternate query. 

### System Design Practice
Using the ByteByteGo framework, sketch the high-level design for:
- "Design a system that continuously monitors an AWS environment and generates compliance reports"
- Identify: functional requirements, non-functional requirements, API design, data model, high-level architecture

System Design: Continuous AWS Compliance Monitor + Report Generator                                                                                                                                                                                                                                                                                                                                                                         
                                                                                                                                                                                                                                                                                                                                                                                                                                              
  ---                                                                                                                                                                                                                                                                                                                                                                                                                                         
  Functional Requirements                 
  - Collect security findings from AWS services (Security Hub, Config, CloudTrail, GuardDuty, IAM) on a schedule                                                                                                                                                                                                                                                                                                                              
  - Map raw findings to NIST 800-53 control IDs                                                                                                                                                                                                                                                                                                                                                                                               
  - Generate downloadable PDF evidence reports per control family                                                                                                                                                                                                                                                                                                                                                                             
  - Detect drift (a control that was passing is now failing)                                                                                                                                                                                                                                                                                                                                                                                  
  - Expose a dashboard showing compliance posture over time                                                                                                                                                                                                                                                                                                                                                                                 
  - Trigger on-demand scans

  Non-Functional Requirements
  - Eventually consistent is acceptable — findings don't need to be real-time to the second
  - Idempotent scans — running twice produces the same result
  - Reports must be tamper-evident (timestamped, immutable after generation)
  - Single-tenant MVP, multi-tenant ready in data model
  - Target: <500 resources per account, <5 min scan-to-report latency

  ---
  API Design
  POST /scans                          # trigger scan
  GET  /scans/{scan_id}                # scan status + summary
  GET  /controls                       # all controls with pass/fail status
  GET  /controls/{control_id}/evidence # raw evidence for a control
  GET  /reports/{report_id}/download   # fetch PDF
  GET  /posture/history                # compliance score over time
  GET  /drift                          # list of controls that changed status

  ---
  Data Model

  Single-table DynamoDB design:

  PK                          SK                        Attributes
  SCAN#{scan_id}              META                      status, started_at, account_id
  SCAN#{scan_id}              FINDING#{finding_id}      service, severity, control_ids[], raw_json
  CONTROL#{control_id}        SCAN#{timestamp}          status (PASS/FAIL), evidence_s3_key
  DRIFT#{date}                EVENT#{ts}#{control_id}   prev_status, new_status, scan_id
  REPORT#{report_id}          META                      s3_key, scan_id, generated_at

  GSI-1: FAMILY#{family_code} → SCAN#{timestamp}   (query all AC controls over time)
  GSI-2: SEVERITY#{level}     → SCAN#{timestamp}   (query all HIGH findings)

  S3 for blobs: raw/{scan_id}/, mapped/{scan_id}/{family}/, reports/{scan_id}/

  ---
  High-Level Architecture

  EventBridge (rate 6h)
          │
          ▼
    collector_lambda          ←── POST /scans (on-demand)
    (boto3: SecurityHub,
     Config, IAM,
     CloudTrail, GuardDuty)
          │
          ├── raw findings → S3 raw/
          └── findings     → DynamoDB SCAN#{id}/FINDING#{}
                                  │
                                  ▼
                         mapping_lambda
                         (NIST control rules JSON)
                                  │
                                  ├── mapped evidence → S3 mapped/
                                  └── control status → DynamoDB CONTROL#{}
                                          │
                            ┌─────────────┼──────────────┐
                            ▼             ▼               ▼
                     pdf_lambda    drift_lambda     posture_lambda
                     (ReportLab)   (compare prev    (aggregate
                          │         vs current)      scores)
                          ▼               │
                     S3 reports/     DynamoDB DRIFT#{}
                          │         SNS → email alert
                          ▼
                 API Gateway → dashboard_lambda → React SPA (S3 + CloudFront)

  ---
  Key Design Decisions
  - Serverless — no EC2 to manage; Lambda + EventBridge handles scheduling and fan-out
  - S3 as the evidence store — immutable, cheap, auditor can be given read-only presigned URLs directly
  - DynamoDB single-table — write patterns (per scan) and read patterns (per control over time) both served without joins
  - Async pipeline — collector → mapper → PDF are decoupled; each Lambda writes its output and the next picks it up, so a PDF generation failure doesn't re-run the collection

---

## Next Up: Phase 1 — AWS Data Collection Pipeline

In the next notebook, you'll write actual boto3 code to pull data from every AWS security service, understand the data formats, and build the collector Lambda function.

**File:** `phase-1-data-collection/01-theory-aws-security-apis.ipynb`